# 02 - Multi-Grid Co-Simulation: pandapower, PyPSA & OpenDSS

This notebook demonstrates heterogeneous power grid co-simulation across transmission, distribution, and market dispatch layers:

1. **pandapower**: Detailed AC power flow and voltage constraint monitoring on a distribution feeder.
2. **PyPSA**: High-level regional economic dispatch and transmission constraints.
3. **OpenDSS**: Multiphase unbalanced 3-phase feeder modeling.
4. **Automatic Unit Resolution**: Seamlessly exchanging variables between `kW` and `MW`.


In [ ]:
from IPython.display import display
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world

# Check optional library installations
try:
    import pandapower as pp
    import pandapower.networks as pn
    has_pp = True
except ImportError:
    has_pp = False

try:
    import pypsa
    has_pypsa = True
except ImportError:
    has_pypsa = False

print(f"pandapower available: {has_pp}")
print(f"PyPSA available     : {has_pypsa}")


## Step 1: Build a Distribution Grid in pandapower

We create a medium-voltage feeder in pandapower with a commercial load and solar generator.


In [ ]:
net = pp.create_empty_network(name="commercial_microgrid")
b1 = pp.create_bus(net, vn_kv=20.0, name="SubstationBus")
b2 = pp.create_bus(net, vn_kv=20.0, name="IndustrialParkBus")
b3 = pp.create_bus(net, vn_kv=20.0, name="SolarFarmBus")

pp.create_ext_grid(net, b1, name="GridSlack")
pp.create_line(net, b1, b2, length_km=2.5, std_type="NAYY 4x150 SE", name="MainFeeder")
pp.create_line(net, b2, b3, length_km=1.2, std_type="NAYY 4x150 SE", name="SolarTieline")

pp.create_load(net, b2, p_mw=1.5, q_mvar=0.3, name="FactoryLoad")
pp.create_sgen(net, b3, p_mw=0.0, q_mvar=0.0, name="SolarPlant")
print("pandapower network configured!")


## Step 2: Build a Regional Dispatch Market in PyPSA

We create an upstream PyPSA market network to evaluate regional dispatch costs and tie-line import power.


In [ ]:
n_pypsa = pypsa.Network()
n_pypsa.add("Bus", "RegionalGrid", v_nom=110.0)
n_pypsa.add("Bus", "LocalSubstation", v_nom=110.0)
n_pypsa.add("Line", "Tieline", bus0="RegionalGrid", bus1="LocalSubstation", x=0.02, r=0.005, s_nom=100.0)
n_pypsa.add("Generator", "ThermalPlant", bus="RegionalGrid", p_nom=200.0, marginal_cost=45.0)
n_pypsa.add("Load", "RegionalLoad", bus="LocalSubstation", p_set=50.0)
print("PyPSA market network configured!")


## Step 3: Couple Grids in energysim World

We couple pandapower and PyPSA together on a 4-hour co-simulation timeline with automatic unit scaling (`kW` $\leftrightarrow$ `MW`).


In [ ]:
sim = world(start_time=0, stop_time=14400, t_macro=900, power_unit="kW")

# Add pandapower AC power flow
sim.add_powerflow(
    sim_name="mv_grid",
    model=net,
    inputs=["FactoryLoad.p_mw", "SolarPlant.p_mw"],
    outputs=["IndustrialParkBus.vm_pu", "SolarFarmBus.vm_pu", "GridSlack.p_mw"],
    step_size=900,
    mode="ac"
)

# Add PyPSA economic dispatch
sim.add_pypsa(
    sim_name="upstream_market",
    model=n_pypsa,
    inputs=["RegionalLoad.P"],
    outputs=["ThermalPlant.P", "LocalSubstation.V"],
    step_size=900,
    mode="ac"
)

# Dynamic solar feed-in
sim.add_signal("solar_profile", lambda t: 2000.0 * min(1.0, t / 7200.0), units={"y": "kW"})
sim.add_connections({
    "solar_profile.y": "mv_grid.SolarPlant.p_mw"
})

print("Multi-grid co-simulation canvas ready!")


## Step 4: Run Simulation and Analyze Power Flow Results


In [ ]:
sim.simulate()
res = sim.last_results

df_grid = res["mv_grid"]
display(df_grid)
